# SALBA ML System - Notebook 3: Model Training & Evaluation

## Objective
Train, evaluate, and compare ML models for disaster prediction.

## Models
1. **Disaster Classifier** - Random Forest (5 classes)
2. **Severity Predictor** - XGBoost (4 classes)
3. **Prank Detector** - Logistic Regression (binary)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries loaded")

In [ ]:
# Load data
df = pd.read_csv('data/training_data.csv')
print(f"Loaded {len(df)} records")

# Prepare features
feature_cols = [
    'text_length', 'word_count', 'has_urgency_keywords', 'has_prank_keywords',
    'latitude', 'longitude', 'hour', 'month'
]

X = df[feature_cols].fillna(0)
y_disaster = df['disaster_type']
y_severity = df['severity']
y_prank = df['is_false_alarm'].astype(int) if 'is_false_alarm' in df.columns else df['has_prank_keywords'].astype(int)

print(f"Features: {len(feature_cols)}")
print(f"Samples: {len(X)}")

In [ ]:
# Encode labels
le_disaster = LabelEncoder()
le_severity = LabelEncoder()

y_disaster_encoded = le_disaster.fit_transform(y_disaster)
y_severity_encoded = le_severity.fit_transform(y_severity)
y_prank_encoded = y_prank.values

print("✅ Labels encoded")
print(f"\nDisaster types: {list(le_disaster.classes_)}")
print(f"Severity levels: {list(le_severity.classes_)}")

In [ ]:
# Train-test split
X_train, X_test, y_dis_train, y_dis_test = train_test_split(
    X, y_disaster_encoded, test_size=0.2, random_state=42, stratify=y_disaster_encoded
)

_, _, y_sev_train, y_sev_test = train_test_split(
    X, y_severity_encoded, test_size=0.2, random_state=42, stratify=y_severity_encoded
)

_, _, y_prank_train, y_prank_test = train_test_split(
    X, y_prank_encoded, test_size=0.2, random_state=42, stratify=y_prank_encoded
)

print(f"✅ Data split: {len(X_train)} train, {len(X_test)} test")
print(f"Train: {len(X_train)} | Test: {len(X_test)}")

In [ ]:
# MODEL 1: DISASTER CLASSIFIER (Random Forest)
print("\n" + "="*60)
print("MODEL 1: DISASTER CLASSIFIER (Random Forest)")
print("="*60)

rf_model = RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_dis_train)

y_dis_pred = rf_model.predict(X_test)
acc_rf = accuracy_score(y_dis_test, y_dis_pred)

print(f"✅ Model trained")
print(f"Accuracy: {acc_rf:.4f}")
print(f"\nClassification Report:")
print(classification_report(y_dis_test, y_dis_pred, target_names=le_disaster.classes_))

In [ ]:
# Feature importance for disaster classifier
feature_importance_rf = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance_rf['feature'], feature_importance_rf['importance'])
plt.xlabel('Importance')
plt.title('Random Forest - Feature Importance for Disaster Classification', fontweight='bold')
plt.tight_layout()
plt.show()

print("\nTop Features:")
print(feature_importance_rf.head(10).to_string(index=False))

In [ ]:
# Confusion matrix for disaster classifier
from sklearn.metrics import confusion_matrix

cm_rf = confusion_matrix(y_dis_test, y_dis_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Blues',
            xticklabels=le_disaster.classes_,
            yticklabels=le_disaster.classes_)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix - Disaster Classifier', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# MODEL 2: SEVERITY PREDICTOR (XGBoost)
print("\n" + "="*60)
print("MODEL 2: SEVERITY PREDICTOR (XGBoost)")
print("="*60)

xgb_model = XGBClassifier(n_estimators=100, max_depth=7, learning_rate=0.1, random_state=42)
xgb_model.fit(X_train, y_sev_train)

y_sev_pred = xgb_model.predict(X_test)
acc_xgb = accuracy_score(y_sev_test, y_sev_pred)

print(f"✅ Model trained")
print(f"Accuracy: {acc_xgb:.4f}")
print(f"\nClassification Report:")
print(classification_report(y_sev_test, y_sev_pred, target_names=le_severity.classes_))

In [ ]:
# Confusion matrix for severity predictor
cm_xgb = confusion_matrix(y_sev_test, y_sev_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm_xgb, annot=True, fmt='d', cmap='Oranges',
            xticklabels=le_severity.classes_,
            yticklabels=le_severity.classes_)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix - Severity Predictor', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# MODEL 3: PRANK DETECTOR (Logistic Regression)
print("\n" + "="*60)
print("MODEL 3: PRANK/FALSE ALARM DETECTOR (Logistic Regression)")
print("="*60)

lr_model = LogisticRegression(random_state=42, max_iter=1000)
lr_model.fit(X_train, y_prank_train)

y_prank_pred = lr_model.predict(X_test)
acc_lr = accuracy_score(y_prank_test, y_prank_pred)

print(f"✅ Model trained")
print(f"Accuracy: {acc_lr:.4f}")
print(f"\nClassification Report:")
print(classification_report(y_prank_test, y_prank_pred, target_names=['Legitimate', 'False Alarm']))

# ROC-AUC for binary classification
try:
    y_prank_proba = lr_model.predict_proba(X_test)[:, 1]
    auc_lr = roc_auc_score(y_prank_test, y_prank_proba)
    print(f"ROC-AUC Score: {auc_lr:.4f}")
except:
    print("ROC-AUC not available")

In [ ]:
# Confusion matrix for prank detector
cm_lr = confusion_matrix(y_prank_test, y_prank_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Greens',
            xticklabels=['Legitimate', 'False Alarm'],
            yticklabels=['Legitimate', 'False Alarm'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix - Prank Detector', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# MODEL COMPARISON
print("\n" + "="*60)
print("MODEL COMPARISON")
print("="*60)

comparison = pd.DataFrame({
    'Model': ['Disaster Classifier (RF)', 'Severity Predictor (XGB)', 'Prank Detector (LR)'],
    'Accuracy': [acc_rf, acc_xgb, acc_lr],
    'Type': ['Multi-class', 'Multi-class', 'Binary'],
    'Classes': [len(le_disaster.classes_), len(le_severity.classes_), 2]
})

print(comparison.to_string(index=False))

plt.figure(figsize=(10, 5))
plt.bar(comparison['Model'], comparison['Accuracy'] * 100, color=['steelblue', 'coral', 'lightgreen'])
plt.ylabel('Accuracy (%)')
plt.title('Model Comparison - Accuracy', fontweight='bold')
plt.ylim([0, 105])
for i, v in enumerate(comparison['Accuracy'] * 100):
    plt.text(i, v + 2, f'{v:.1f}%', ha='center', fontweight='bold')
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Save models
import joblib
import os

models_dir = 'models'
os.makedirs(models_dir, exist_ok=True)

joblib.dump(rf_model, f'{models_dir}/disaster_classifier.pkl')
joblib.dump(xgb_model, f'{models_dir}/severity_predictor.pkl')
joblib.dump(lr_model, f'{models_dir}/prank_detector.pkl')
joblib.dump(le_disaster, f'{models_dir}/disaster_encoder.pkl')
joblib.dump(le_severity, f'{models_dir}/severity_encoder.pkl')

print("✅ Models saved:")
print(f"  - {models_dir}/disaster_classifier.pkl")
print(f"  - {models_dir}/severity_predictor.pkl")
print(f"  - {models_dir}/prank_detector.pkl")
print(f"  - {models_dir}/disaster_encoder.pkl")
print(f"  - {models_dir}/severity_encoder.pkl")

In [ ]:
# PREDICTIONS EXAMPLES
print("\n" + "="*60)
print("EXAMPLE PREDICTIONS")
print("="*60)

# Take first 5 test samples
examples = X_test.iloc[:5]

print("\nSample predictions:")
for idx, (i, sample) in enumerate(examples.iterrows(), 1):
    print(f"\n--- Sample {idx} ---")
    print(f"Features: {sample.to_dict()}")
    
    # Disaster prediction
    dis_pred = rf_model.predict([sample.values])[0]
    dis_proba = rf_model.predict_proba([sample.values])[0].max()
    print(f"🎯 Disaster: {le_disaster.classes_[dis_pred]} (confidence: {dis_proba:.2%})")
    
    # Severity prediction
    sev_pred = xgb_model.predict([sample.values])[0]
    sev_proba = xgb_model.predict_proba([sample.values])[0].max()
    print(f"⚠️  Severity: {le_severity.classes_[sev_pred]} (confidence: {sev_proba:.2%})")
    
    # Prank prediction
    prank_pred = lr_model.predict([sample.values])[0]
    prank_proba = lr_model.predict_proba([sample.values])[0].max()
    prank_label = "False Alarm" if prank_pred == 1 else "Legitimate"
    print(f"✓ Verification: {prank_label} (confidence: {prank_proba:.2%})")

In [ ]:
# TRAINING SUMMARY
print("\n" + "="*60)
print("TRAINING SUMMARY")
print("="*60)
print(f"✅ All models trained successfully")
print(f"✅ Train size: {len(X_train)} samples")
print(f"✅ Test size: {len(X_test)} samples")
print(f"\n📊 Model Accuracies:")
print(f"   Disaster Classifier: {acc_rf:.2%}")
print(f"   Severity Predictor: {acc_xgb:.2%}")
print(f"   Prank Detector: {acc_lr:.2%}")
print(f"\n💾 Models saved to /models/ directory")
print(f"\nNext steps:")
print(f"1. Run 'python app.py' to start Flask API")
print(f"2. Test predictions via /api/ml endpoints")
print(f"3. Integrate with Node.js backend")